<a href="https://colab.research.google.com/github/vinipi/ailead_gpumanagement/blob/main/exercise_3_fine_tune_llamba8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install transformers peft bitsandbytes accelerate datasets trl gradio --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.2 MB/s eta 0:00:00


In [ ]:
import torch
import gc
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login

# Paste your Hugging Face read token when asked.
login()

torch.cuda.empty_cache() # Added this line to try and free up memory
gc.collect() # Added this line to try and free up memory

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    print("Warning: no GPU found. Training needs an NVIDIA GPU.")

print("Device:", device)

# The model you will fine-tune.
# MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

# No Llama access yet? Use this open copy instead. Just remove the # on the next line.
MODEL_NAME = "NousResearch/Meta-Llama-3-8B-Instruct"


In [ ]:
# These settings load the model in 4-bit. They are the standard QLoRA settings.
# nf4 is a number format made for model weights. Double quant saves a little more.
compute_dtype = torch.float16  # Use torch.float16 on a T4. Newer cards can use torch.bfloat16.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# YOUR CODE HERE
# Load the model. Use AutoModelForCausalLM.from_pretrained with these three arguments:
#   MODEL_NAME
#   quantization_config=bnb_config
#   device_map="auto"
# Store it in a variable called model.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False


# YOUR CODE HERE
# Print the GPU memory the model uses, in GB.
# Hint: torch.cuda.memory_allocated() / 1e9
print(f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
support_pairs = []
support_pairs.append(("How long does a typical home solar installation take?",
                      "Most Heliora installations finish in one day. We arrive in the morning and your system is usually running by evening. Larger roofs can take a second day, and we tell you in advance if that applies to you."))
support_pairs.append(("Why did my electricity bill change after going solar?",
                      "Your bill now has two parts: a smaller charge from the grid and the savings from the power your panels make. In sunny months you may even build a credit. We are happy to walk through your first bill line by line."))
support_pairs.append(("What does the Heliora warranty cover?",
                      "Heliora covers your panels and inverter for 25 years, and the installation workmanship for 10 years. If anything underperforms in that time, we repair or replace it at no cost to you."))
support_pairs.append(("Can I add a battery to my system later?",
                      "Yes. Every Heliora system is battery ready. You can add storage at any time, and we size it to match how much backup power you want during an outage."))
support_pairs.append(("What happens to my panels in a power outage?",
                      "For safety, a standard grid system pauses during an outage. If you have a Heliora battery, your home keeps running on stored power. Without a battery, power returns when the grid comes back."))
support_pairs.append(("How do I see how much energy my panels produce?",
                      "The Heliora app shows your production in real time, day by day and month by month. You can see your savings, check system health, and get an alert if anything needs attention."))

SYSTEM_PROMPT = "You are Heliora Energy's friendly support assistant. Answer customer questions about home solar clearly, in a warm and honest tone. Keep answers short and accurate."

texts = []
for pair in support_pairs:
    question = pair[0]
    approved_answer = pair[1]

    # YOUR CODE HERE
    # Build a list called messages with three dicts:
    #   system    -> SYSTEM_PROMPT
    #   user      -> question
    #   assistant -> approved_answer
    # Store it in a variable called messages.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": approved_answer},
    ]


    # YOUR CODE HERE
    # Turn messages into one text string with the chat template.
    # Use tokenizer.apply_chat_template(messages, tokenize=False).
    # Store it in a variable called formatted, then append formatted to texts.
    formatted = tokenizer.apply_chat_template(messages, tokenize=False)
    texts.append(formatted)

# YOUR CODE HERE
# Build the dataset from texts. Use Dataset.from_dict({"text": texts}).
# Store it in a variable called train_dataset.
# Then print len(train_dataset) and print train_dataset[0]["text"].
train_dataset = Dataset.from_dict({"text": texts})
print(len(train_dataset))
print(train_dataset[0]["text"])


In [ ]:
# Step 1: get the 4-bit model ready for training.
# This freezes the base weights and turns on a memory-saving trick for you.
model = prepare_model_for_kbit_training(model)

# These are good LoRA settings for an 8B model.
# r and lora_alpha set the size of the add-on layers.
# target_modules lists the layers the adapters attach to.
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

# YOUR CODE HERE
# Wrap the model with the LoRA config.
# Use get_peft_model(model, lora_config) and store it back in model.
model = get_peft_model(model, lora_config)

# YOUR CODE HERE
# Print how many weights will train. Use model.print_trainable_parameters().
model.print_trainable_parameters() # Corrected this line

In [ ]:
from trl import SFTTrainer, SFTConfig

# Use 16-bit math that fits your card.
use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16

# These settings are chosen to be light on memory.
# This demo has only six pairs, so we use more epochs to get a real change.
# With a real dataset of hundreds of pairs, use gradient_accumulation_steps=8 and 1 to 3 epochs.
sft_config = SFTConfig(
    output_dir="heliora-llama3-8b-training",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    num_train_epochs=10,
    max_length=512,
    dataset_text_field="text",
    logging_steps=5,
    fp16=use_fp16,
    bf16=use_bf16,
    report_to="none",
)

# YOUR CODE HERE
# Build the trainer. Pass:
#   model=model
#   args=sft_config
#   train_dataset=train_dataset
#   processing_class=tokenizer   (if this errors, use tokenizer=tokenizer instead)
# Store it in a variable called trainer.
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)


# YOUR CODE HERE
# Start training with trainer.train().
trainer.train()


In [ ]:
test_question = "Do you offer a referral discount if I recommend Heliora to a neighbor?"

model.eval()

# Build the prompt: the system message plus the new question.
test_messages = []
test_messages.append({"role": "system", "content": SYSTEM_PROMPT})
test_messages.append({"role": "user", "content": test_question})

prompt_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt")
inputs = inputs.to(model.device)

prompt_length = inputs["input_ids"].shape[1]


# This helper turns the model output into clean text. It keeps only the new part.
def read_new_text(output_ids):
    new_tokens = output_ids[0][prompt_length:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return text


with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)

print("Fine-tuned answer:")
print(read_new_text(output_ids))

with model.disable_adapter():
    with torch.no_grad():
        base_output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)

print("\nBase model answer:")
print(read_new_text(base_output_ids))

print("\nComparison: The fine-tuned answer should use Heliora's short, warm support style and stay close to the company facts. The base answer may be more generic or add details that were not in the training examples.")


In [ ]:
import gradio as gr

model.eval()

def answer(user_message, history):
    messages = []
    messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_message})

    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt")
    inputs = inputs.to(model.device)
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)

    # Turn the output into clean text and return it.
    new_tokens = output_ids[0][prompt_length:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return reply


chat = gr.ChatInterface(fn=answer, title="Heliora Support Assistant")
chat.launch()
